In [1]:
import numpy as np
import pandas as pd

INPUT_ID = 1
trans_df = pd.read_csv(f"./datasets/input_{INPUT_ID}.csv")
print("SIZE:", trans_df.size)
trans_df.head(5)

SIZE: 1833337


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 10:17,22661,8018086D0,146606,81BF2C430,1088.39,US Dollar,1088.39,US Dollar,ACH,0
1,2022/09/01 13:16,336343,80D675B10,336343,80D675B10,11074.08,Canadian Dollar,11074.08,Canadian Dollar,Reinvestment,0
2,2022/09/01 20:30,218587,808D26B30,218587,808D26B30,784.79,Euro,784.79,Euro,Reinvestment,0
3,2022/09/06 10:29,1776,8007B2390,1768,8007FA340,35.05,Euro,35.05,Euro,Cheque,0
4,2022/09/09 01:30,139833,81632BC50,231905,8163371C0,3176.53,US Dollar,3176.53,US Dollar,Cash,0


In [2]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/14 10:53]


In [3]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Series([], Name: Bank, dtype: int64)

In [4]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 61501


In [5]:
# Analyze accounts.
accounts_df = pd.read_csv(f"./datasets/accounts_{INPUT_ID}.csv")
print("SIZE:", accounts_df.shape[0])

SIZE: 101036


In [6]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 33473


In [7]:
ranged_trans_usd_sept_df = trans_usd_sept_1st_df\
    .groupby(["From Bank", "Account"])\
    .filter(lambda x: x.groupby(["To Bank", "Account.1"]).size().size >= 5)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 2715


In [8]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions.sort_values(by=["From Bank"], ascending=True)

,From Bank,Account,To Bank,Account.1,Amount Paid
129839,0,80023FDC0,15640,802479D10,13.35
157369,1,80C4FD5E0,22129,81A975840,31.35
81744,1,8002EF090,1,800B79AF0,2.31
52596,1,80781A690,1,80781A690,11.44
21376,1,803C651A0,11483,80638BBF0,10.48
...,...,...,...,...,...
71638,364278,8177A6970,124222,8177882B0,0.01
137678,366025,81825FF00,249786,81825FEB0,18.61
54089,366025,81825FF00,249786,81825FEB0,18.61
99440,367012,81886C7A0,18270,81884C5A0,0.02


In [9]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
max_amount_bank=max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]].drop_duplicates().sort_values(by="Account", ascending=True)
max_amount_bank

,From Bank,Account,Bank Name,Amount Paid
9054,70,10042B660,Willows Thrift,1.950319e+08
363,1,800056ED0,First Bank of Portland,1.111656e+08
7472,23,800077210,Japan Bank #46,1.099376e+04
20076,3193,80010EEC0,Fieldstone Bank,3.204800e+02
20227,3211,8001219A0,Bank of Lacrosse,3.468000e+02
...,...,...,...,...
64856,313213,81C097970,Bank of Phoenix,3.627600e+02
65678,335288,81C120350,Savings Bank of the Valley,1.738200e+02
65963,349874,81C128270,Regents Federal Bank,1.314500e+02
64705,270657,81C164780,Capital Thrift,4.775000e+01


In [10]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})

print(trans_usd_sept_2nd_with_avg_df.loc[:, ["AVG", "Payment Format"]].drop_duplicates())
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
lower_trans_usd_sept_2nd_with_avg_df=lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]].sort_values(by=["Account", "Amount Paid"], ascending=True)
lower_trans_usd_sept_2nd_with_avg_df

              AVG Payment Format
0    1.273789e+06           Cash
1    3.032019e+05         Cheque
3    3.505398e+03    Credit Card
6    8.380469e+05            ACH
194  8.573435e+05           Wire


,From Bank,Account,Payment Format,Amount Paid
23766,70,10042B660,Cash,0.02
11217,70,10042B660,Cash,0.54
24825,70,10042B660,Cash,0.59
24773,70,10042B660,Cash,0.79
26202,70,10042B660,Cash,0.83
...,...,...,...,...
18108,21438,81BFE33F0,ACH,2707.19
27115,157989,81C027330,ACH,264.45
15218,336418,81C038350,Cheque,250.13
2778,254164,81C055A30,ACH,923.51


In [19]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
print(trans_usd_sept_1st_df[(trans_usd_sept_1st_df["Account.1"] == "807C60BC0")])  
account_pairs_df = accounts_df.merge(trans_usd_sept_1st_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
print(account_pairs_df[(account_pairs_df["To Account"] == "807C60BC0")])  
account_pairs_df = account_pairs_df[(account_pairs_df["size"] >= 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts

Empty DataFrame
Columns: [From Bank, Account, To Bank, Account.1]
Index: []
     From Bank From Account  To Bank To Account  size
368         70    10042B660   116703  807C60BC0     6


,Bank,Account
98,70,10042B660
98,2860,800F04E90
368,116703,807C60BC0


In [12]:
#Conversion dates of period [2022-09-01, 2022-09-05] with base USD
#Bitcoin rates taken from investing.com
#Rest of currencies from api.frankfurter.dev
conversion_rates_records = np.rec.array([
           ('2022/09/01', 1.4644, 5.1805, 1.314 , 0.97999, 6.9   , 1.0002, 0.86272, 3.3535, 79.543, 139.34, 20.189, 60.367, 3.75, 1.,  19793.1),
           ('2022/09/02', 1.4691, 5.2035, 1.3141, 0.98175, 6.9035, 1.0011, 0.86468, 3.3755, 79.719, 140.11, 20.085, 60.427, 3.75, 1., 199999. ),
           ('2022/09/03', 1.4691, 5.2056, 1.3138, 0.98207, 6.9046, 1.0013, 0.86478, 3.3791, 79.75 , 140.17, 20.081, 60.471, 3.75, 1.,  19831.4),
           ('2022/09/04', 1.4695, 5.2082, 1.3139, 0.98219, 6.9047, 1.0013, 0.8649 , 3.3815, 79.754, 140.22, 20.084, 60.461, 3.75, 1.,  19952.7),
           ('2022/09/05', 1.4722, 5.1786, 1.3142, 0.98273, 6.9216, 1.0068, 0.86813, 3.4006, 79.816, 140.49, 20.018, 60.737, 3.75, 1.,  20126.1)],
          dtype=[ ('Date', 'O'), ('Australian Dollar', '<f8'), ('Brazil Real', '<f8'), ('Canadian Dollar', '<f8'), ('Swiss Franc', '<f8'), ('Yuan', '<f8'), ('Euro', '<f8'), ('UK Pound', '<f8'), ('Shekel', '<f8'), ('Rupee', '<f8'), ('Yen', '<f8'), ('Mexican Peso', '<f8'), ('Ruble', '<f8'), ('Saudi Riyal', '<f8'), ('US Dollar', '<f8'), ('Bitcoin', '<f8')])
conversion_rates_df = pd.DataFrame.from_records(conversion_rates_records)
conversion_rates_df = conversion_rates_df.set_index("Date")

In [13]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df.apply(lambda row: row['Amount Paid'] / conversion_rates_df[row['Payment Currency']][row["Timestamp"].split(" ")[0]], axis=1)
trans_sept_1st_wire_or_ach_filtered = trans_sept_1st_wire_or_ach_converted_df[trans_sept_1st_wire_or_ach_converted_df['Amount'] < 1.0]
print("SIZE:", trans_sept_1st_wire_or_ach_filtered.shape[0])

SIZE: 228


In [14]:
from pandas.testing import assert_frame_equal
result_q1 = pd.read_csv(f"./output/q1_output_{INPUT_ID}.csv")
q1_comp1=low_profile_transactions.sort_values(by=["From Bank", "Account"], ascending=True).reset_index(drop=True)
q1_comp2=result_q1.sort_values(by=["From Bank", "Account"], ascending=True).reset_index(drop=True).round(2)

assert_frame_equal(q1_comp1, q1_comp2)

In [15]:
result_q2 = pd.read_csv(f"./output/q2_output_{INPUT_ID}.csv")
q2_comp1=max_amount_bank.sort_values(by=["Amount Paid", "Account"], ascending=True).reset_index(drop=True)
q2_comp2=result_q2.sort_values(by=["Amount Paid", "Account"], ascending=True).reset_index(drop=True).round(2)

assert_frame_equal(q2_comp1[q2_comp2.columns], q2_comp2)

In [16]:
result_q3 = pd.read_csv(f"./output/q3_output_{INPUT_ID}.csv")
q3_comp1=lower_trans_usd_sept_2nd_with_avg_df.sort_values(by=["From Bank", "Account", "Payment Format", "Amount Paid"], ascending=True).reset_index(drop=True)
q3_comp2=result_q3.sort_values(by=["From Bank", "Account", "Payment Format", "Amount Paid"], ascending=True).reset_index(drop=True).round(2)
df = q3_comp1[q3_comp2.columns].merge(q3_comp2, on=q3_comp2.columns.tolist(), how='outer', suffixes=['', '_'], indicator=True)
assert_frame_equal(q3_comp1[q3_comp2.columns], q3_comp2)

In [17]:
result_q4 = pd.read_csv(f'./output/q4_output_{INPUT_ID}.csv').rename(columns={'From Bank': 'Bank'})
q4_comp1 = unique_accounts.sort_values(by=["Bank", "Account"], ascending=True).reset_index(drop=True)
q4_comp2 = result_q4[unique_accounts.columns].sort_values(by=["Bank", "Account"], ascending=True).reset_index(drop=True)
q4_comp1['Account'] = q4_comp1['Account'].astype(str)
q4_comp2['Account'] = q4_comp2['Account'].astype(str)

assert_frame_equal(q4_comp1, q4_comp2)

AssertionError: DataFrame are different

DataFrame shape mismatch
[left]:  (3, 2)
[right]: (2, 2)

In [ ]:
print(result_q4)
q4_comp1

    Bank    Account                         Destinations
0  25526  815D6E610                                  NaN
1  22435  804070470  {'bank=25526 account=815D6E610': 6}
2     70  800333111                                  NaN
3     70  800111222     {'bank=70 account=800333111': 6}


,Bank,Account
0,70,800111222
1,70,800333111


In [ ]:
df = pd.read_csv(f'./output/q5_output_{INPUT_ID}.csv', header=None)
valor_unico = df.iloc[0, 0]
print(trans_sept_1st_wire_or_ach_filtered.shape[0]==valor_unico) 

True
